In [17]:
import os
import pandas as pd

bad_id_path = "../data/processed/metadata/bad_bubble_id.txt"

os.makedirs(os.path.dirname(bad_id_path), exist_ok=True)

if not os.path.exists(bad_id_path):
    bad_df = pd.DataFrame(columns=["CATALOGUE", "ID"])
    bad_df.to_csv(bad_id_path, index=False)

print("Bad bubble ID file ready:", bad_id_path)

Bad bubble ID file ready: ../data/processed/metadata/bad_bubble_id.txt


In [18]:
catalogues = {
    "A": "../data/raw/catalogue/jwst_bubble_properties_A.txt",
    "B": "../data/raw/catalogue/jwst_bubble_properties_B_fixed.txt",
    "C": "../data/raw/catalogue/jwst_bubble_properties_C.txt",
}

all_catalogues = []

for catalogue_label, path in catalogues.items():
    df = pd.read_csv(path)
    df["CATALOGUE"] = catalogue_label
    all_catalogues.append(df)

df_all = pd.concat(all_catalogues, ignore_index=True)

# Creating a global unique ID
df_all.insert(0, "GLOBAL_ID", range(1, len(df_all) + 1))

print("Total bubbles before cleaning:", len(df_all))
print(df_all[["CATALOGUE", "ID", "AVG_RAD_PC"]].head())

Total bubbles before cleaning: 3318
  CATALOGUE  ID  AVG_RAD_PC
0         A   1          86
1         A   2          24
2         A   3          42
3         A   4          25
4         A   5          24


In [19]:
bad_df = pd.read_csv(bad_id_path)

print("Bad bubbles listed:", len(bad_df))
print(bad_df.head())

Bad bubbles listed: 78
  CATALOGUE    ID
0         A  1506
1         A  1450
2         A  1161
3         A  1222
4         A  1332


In [20]:
df_clean = df_all.merge(
    bad_df,
    on=["CATALOGUE", "ID"],
    how="left",
    indicator=True
)

df_clean = df_clean[df_clean["_merge"] == "left_only"].drop(columns=["_merge"])

print("Total bubbles after cleaning:", len(df_clean))
print("Removed bubbles:", len(df_all) - len(df_clean))

Total bubbles after cleaning: 3259
Removed bubbles: 59


In [21]:
collective_path = "../data/processed/metadata/catalogue_collective.txt"

df_clean.to_csv(collective_path, index=False)

print("Saved clean collective catalogue:", collective_path)

Saved clean collective catalogue: ../data/processed/metadata/catalogue_collective.txt


In [22]:
# // ADD BAD BUBBLE VALIDATION and have it verified